# 00 — Overview

Welcome to the analysis suite.  Every notebook here reads the CSV
reports written by `Experiment.run()` under `reports/<exp_id>/`:

* `episode_metrics.csv` — one row per episode (training, eval, …).
* `step_metrics.csv`    — one row per env step.
* `config.json`         — the env + agent + experiment config snapshot.

This first notebook shows:

1. Which runs are available on disk.
2. How to load one.
3. The high-level structure of the resulting DataFrames.
4. A one-glance summary of the run.

In [ ]:
import sys, pathlib
# Make ``utils.py`` importable whether the notebook is opened from
# project root or from analysis/.
_here = pathlib.Path.cwd()
for cand in (_here, _here / "analysis", _here.parent):
    if (cand / "utils.py").exists():
        sys.path.insert(0, str(cand))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    list_runs, latest_run, load_run, load_runs,
    group_columns, group_columns_by_prefix, strip_prefix, rolling_mean,
)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

## Available runs

In [ ]:
runs_df = list_runs()
runs_df

## Load the latest run

In [ ]:
# Pick a run.  ``latest_run()`` returns the most recently modified
# experiment in ``reports/``.  Override ``EXP_ID = "..."`` to analyse
# a specific one.
EXP_ID = latest_run()
print("Analysing:", EXP_ID)

ep, st, cfg = load_run(EXP_ID)
print(f"  episode_metrics.csv: shape={ep.shape}")
print(f"  step_metrics.csv:    shape={st.shape if st is not None else 'absent'}")

## Episode metrics — columns by prefix

The dataframe is wide.  Columns are namespaced with a prefix that
tells you what they are:

| Prefix | Meaning |
|---|---|
| `metric/` | Episode-level KPIs (`metric/mttr`, `metric/total_breakdowns`, …) |
| `step_mean/` | Mean of each per-step metric over the episode |
| `reward_sum/` / `reward_mean/` | Total + mean per-step reward components |
| `update/` | Agent-update metrics (PPO loss, KL, …) |
| `assignments/` | Per-tech repair-assignment count |
| `machine_maintenance/` / `machine_production/` / `machine_breakdowns/` | Per-machine episode stats |

In [ ]:
groups = group_columns_by_prefix(ep)
for prefix in sorted(groups):
    print(f"  {prefix + '/':<25} {len(groups[prefix])} columns")

## Step metrics — head

In [ ]:
if st is not None and not st.empty:
    display(st.head(5))
else:
    print("no step_metrics.csv for this run")

## High-level summary

In [ ]:
summary = (
    ep.groupby("phase")
      .agg(
          n_episodes=("episode", "count"),
          mean_return=("return", "mean"),
          last_return=("return", "last"),
          mean_length=("length", "mean"),
          total_wall_clock_s=("wall_clock_seconds", "sum"),
      )
      .round(3)
)
summary

## Config snapshot

In [ ]:
print("agent:    ", cfg.get("agent", {}).get("agent_type"))
print("seed:     ", cfg.get("experiment", {}).get("seed"))
print("n_eps:    ", cfg.get("experiment", {}).get("n_episodes"))
print("obs_repr: ", cfg.get("env", {}).get("gym", {}).get("observation_representation"))
print("obs_mode: ", cfg.get("env", {}).get("gym", {}).get("observation_mode"))
print("randomized:", cfg.get("env", {}).get("randomized_scenario", {}).get("enabled"))